# Pathway networks along the PT: scoranet's ideas against the positional F-test and clustering (notebook 93)

**The question.** scoranet (single-cell over-representation analysis networks) links enriched pathways whose
driving genes overlap, groups them with Leiden and names each group by a shared parent term. Can its ideas, with the
PT axis added, find or organise pathway biology better than the positional F-test or the clustering method?

**The frozen protocol** is `docs/results/pt-pathway-networks.md` (commit dbefcba), fixed before any fit here. It asks:
- **Q1 · calling:** is position-split ORA (scoranet's gene-first ORA with its up/down split extended to up/down ×
  segment; row N) specific, stable, unexposed and robust to hub genes, against the positional F-test (F) and
  clustering (C)?
- **Q2 · organising:** on each method's calls, does the network (NET), or a network linking calls by where along the
  PT their difference lies (NET-pos), group them into fewer programs that are more coherent, replicate across
  specimens and survive relabeling and a change of reconstruction, against 75's overlap programs (OVL) and 71's
  redundancy clusters (RED)?
- **Q3 · discovery:** which programs do the methods find, where along the PT, and are they different programs or the
  same programs through different sets?

Everything is descriptive: two control mice, two AKI mice and two cortex sections of one human donor. Specimens are
the replicates. Notebook 71's gene models are read from its cache and never refit.

**Sections.**
1. Setup, inputs and guards; segments and gene shapes.
2. The three methods' calls and driver genes, including row N.
3. Q1: calling criteria and verdicts.
4. The networks, names and positional attributes.
5. Q2: grouping criteria and verdicts.
6. Q3: program tables and the union network.
7. Application: notebook 91's injury screen.
8. Figures.
9. How the protocol was read, summary and run record.

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import random
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
package_root = results_root / 'package_pseudospace'
upstream13 = results_root / 'minimal_pt_scfates'                        # the ortholog map
T71 = results_root / 'pt_pathways_package_pseudospace'                  # notebook 71: read only
T69 = results_root / 'pt_pathways_without_mito'                         # notebook 69: read only (reconstruction check)
T75 = results_root / 'pt_package_pathway_methods'                       # notebook 75: programs and probe exposure
T91 = results_root / 'pt_positional_decomposition'                      # notebook 91: the injury screen
nb38 = results_root / 'pt_literature_deep'                              # probe counts
coordinate_file = results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv'        # 69's SCF13
nb12_universe_file = results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv'   # 69's AKI pool
HIER = data_root / 'external' / 'pathway_hierarchies'
output = results_root / 'pt_pathway_networks'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
DATASETS = ['human_vs_mouse', 'aki_vs_control']
hvm, aki = DATASETS
SHORT_DS = {hvm: 'hvm', aki: 'aki'}
EXPORT = {ds: package_root / SHORT_DS[ds] / 'pt_units.h5ad' for ds in DATASETS}
FINGERPRINTS = {hvm: 'efafc9ac2e99', aki: 'bb4a80c1b484'}                # confirmed pass-1 labels (notebook 70)
LIBRARIES = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
HIERARCHY_SHA = {'ReactomePathwaysRelation.txt': 'fd49a624d80c14eb37ae57a02e141d574d5ede3f60022bb99edbd909448a3f1e',
                 'ReactomePathways.txt': 'f6d7a2bf89b5bcfe0250a0bc7f51bff94641447911712b8ff129f5b55e52df3a',
                 'br08901.keg': '86f3c0037e72b96e78b6ea379fc57d234c73b9bfa65f64c191a8d70e8db2a406'}
REQUIRED = [*EXPORT.values(), upstream13 / 'ortholog_map_used.csv', nb38 / 'probe_counts.csv', coordinate_file,
            nb12_universe_file, *[HIER / f for f in HIERARCHY_SHA],
            *[T71 / 'tables' / f'{kind}_{ds}.{ext}' for ds in DATASETS
              for kind, ext in (('universe', 'csv'), ('gene_sets', 'json'), ('gene_statistics', 'csv'),
                                ('redundancy_clusters', 'csv'), ('s1_pathway_tests', 'csv'), ('unit_matching', 'csv'))],
            *[T71 / 'tables' / f for f in ('inputs.csv', 'method_table.csv', 'compare69_pathways_summary.csv')],
            *[T69 / 'tables' / f'{kind}_{ds}.csv' for ds in DATASETS for kind in ('gene_statistics', 'redundancy_clusters')],
            *[T69 / 'tables' / f'pathway_tests_{ds}_cameraPR_species.csv' for ds in DATASETS],
            *[T75 / 'tables' / f for f in ('probe_exposure.csv', 'method_table.csv', 'programs_human_vs_mouse.csv',
                                            'programs_aki_vs_control.csv')],
            T91 / 'tables' / 'i5_library_sets.csv', T91 / 'tables' / 'i_genes.csv.gz',
            *[data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json' for lib in LIBRARIES]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 38, 69, 70, 71, 75 and 91 first)')

NOTEBOOK_LOGIC_VERSION = '93.networks.1'   # Bump when a cached calculation changes.
MT_PREFIX = 'mt-'
ALPHA = .05                                      # gene q, set q and call threshold
HALF_RANGE = .5                                  # row N: half-range of the positional difference, log2 (scoranet's 0.5)
JACCARD_EDGE, RESOLUTION, LEIDEN_SEED = .1, 1., 1234   # scoranet's network defaults
PROFILE_EDGE = .8                                # NET-pos: call-profile Pearson r
TOP_SHARE, PROGRAM_CUT, MIN_DRIVERS = .10, .5, 3 # 75's overlap programs and contributing genes
N_NULL, NULL_SEED = 1000, 93
N_HUB = 10
MIN_SHARE, MAX_DESC_SHARE, MAX_PICKS = .15, .04, 2   # scoranet's ancestor rule (1,000 of about 26,000 GO BP terms)
SCORE_THRESHOLD = 1.
SWEEP_THRESHOLDS, SWEEP_RESOLUTIONS, SWEEP_CUTS = (.05, .1, .2, .3), (.5, 1., 2.), (.3, .4, .5, .6, .7)
CALL_DIRECTIONS = {'swap rule': 1, 'reconstruction Jaccard': 1, 'probe exposure': -1, 'hub robustness': 1}
CALL_TOL = {'reconstruction Jaccard': .05, 'probe exposure': .05, 'hub robustness': .05}
GROUP_DIRECTIONS = {'programs per call': -1, 'coherence z': 1, 'replication z': 1, 'reconstruction ARI': 1, 'program NCDR': -1}
GROUP_TOL = {'programs per call': .02, 'coherence z': .5, 'replication z': .5, 'reconstruction ARI': .05, 'program NCDR': .05}
SET_SIZES = (10, 300)
CAMERA_COR = .01
DF1, DF2 = 6, {'condition': 12, 'relabeling': 6}
GRID_POINTS = 101
EXTERNAL_COMMIT = '59da04b'
METHODS = {'F': 'positional F-test', 'C': 'clustering', 'N': 'position-split ORA'}
GROUPINGS = {'NET': 'network (gene overlap)', 'NET-pos': 'network (position)', 'OVL': 'overlap programs (75)',
             'RED': 'redundancy clusters (71)'}
LIST_KEYS = [(d, s) for d in ('up', 'down') for s in ('S1', 'S2', 'S3')]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from importlib.metadata import distribution, version
from IPython.display import display

direct_url = json.loads(distribution('pseudospace-reconstruction').read_text('direct_url.json') or '{}')
EXTERNAL = {'version': version('pseudospace-reconstruction'),
            'commit': direct_url.get('vcs_info', {}).get('commit_id', 'unknown')}
assert EXTERNAL['commit'].startswith(EXTERNAL_COMMIT), f'pseudospace_reconstruction is not commit {EXTERNAL_COMMIT}: {EXTERNAL}'

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
METHOD_COLOR = {'F': '#1b8a5a', 'C': '#eda100', 'N': '#2a78d6'}
GROUPING_COLOR = {'NET': '#2a78d6', 'NET-pos': '#9C93D3', 'OVL': '#1b8a5a', 'RED': '#8a8985'}
REFERENCE_COLOR, CASE_COLOR = '#0072B2', '#D55E00'
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


print('Results folder:', output, '· pseudospace_reconstruction', EXTERNAL['version'], EXTERNAL['commit'][:7])

### 1.1 · Inputs and guards

Inputs are built as notebook 75 builds them (`package_inputs`, mt genes out of the library and the universe).

**Guards (not cached):**
- G1: the rebuilt universe, sets and units equal notebook 71's, and the cached gene models reproduce 71's T_spatial
  and z (75's G1–G3).
- G6: the hierarchy files match the frozen sha256.

In [ ]:
from pseudospace.package_inputs import aki_inputs, hvm_inputs, read_pt_units
from pseudospace.pathway_pipelines import GeneModel, camera_pr, common_grid, moderated_f, partition_groups

t0 = time.perf_counter()
guard_log = []
for name, sha in HIERARCHY_SHA.items():                                                                      # G6
    assert hashlib.sha256((HIER / name).read_bytes()).hexdigest() == sha, f'G6: {name} changed'
guard_log.append('G6: the Reactome and KEGG hierarchy files match the frozen sha256')

orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith(MT_PREFIX))
assert len(MT) == 13, MT
UNITS = {ds: read_pt_units(EXPORT[ds], SHORT_DS[ds]) for ds in DATASETS}
for ds, units in UNITS.items():
    prov = units.uns['provenance']
    assert str(prov['notebook_logic_version']).startswith('70.'), prov['notebook_logic_version']
    assert str(prov['package']['commit']).startswith(EXTERNAL_COMMIT), prov['package']
    assert prov['fingerprints']['pass1']['membership_sha1'] == FINGERPRINTS[ds], ds
kept_h = hvm_inputs(UNITS[hvm], data_root, orthologs)
built = {hvm: hvm_inputs(UNITS[hvm], data_root, orthologs, exclude_genes=MT)}
built[aki] = aki_inputs(UNITS[aki], data_root, orthologs, kept_h['genes'], exclude_genes=MT)
del kept_h
TITLES = {hvm: ('Human vs mouse PT', 'human', 'mouse'), aki: ('AKI vs control mouse PT', 'AKI', 'control')}
INPUTS71 = pd.read_csv(T71 / 'tables' / 'inputs.csv', index_col=0)
DATA = {}
for ds in DATASETS:
    new = built[ds]
    title, case_name, reference_name = TITLES[ds]
    universe71 = pd.read_csv(T71 / 'tables' / f'universe_{ds}.csv').gene.astype(str).tolist()
    sets71 = json.loads((T71 / 'tables' / f'gene_sets_{ds}.json').read_text())
    assert new['genes'] == universe71 and new['gene_sets'] == sets71, 'G1: the universe or sets differ from notebook 71'
    assert int(INPUTS71.loc['units in the common support', title]) == len(new['position'])
    DATA[ds] = {'title': title, 'case_name': case_name, 'reference_name': reference_name,
                'counts': new['counts'].tocsr(), 'position': np.asarray(new['position'], float),
                'specimen': new['specimen'].astype(str),
                'segment': pd.Series(new['segment']).str.replace('PT-', '', regex=False).to_numpy(),
                'case': np.asarray(new['case'], bool), 'genes': list(new['genes']), 'sets': new['gene_sets']}
del built, UNITS


def slug(label):
    return label.replace(':', '_').replace('+', '_')


PARTITIONS, SHORT = {}, {}
for ds, d in DATA.items():
    PARTITIONS[ds] = partition_groups(d['specimen'], d['case'])
    labels = list(PARTITIONS[ds])
    assert labels[0] == 'species'
    SHORT[ds] = dict(zip(labels, ['condition', 'relabeling 1', 'relabeling 2']))
    d['log_mean_count'] = np.log(np.asarray(d['counts'].mean(axis=0)).ravel())
    d['case_specimens'] = sorted(set(d['specimen'][d['case']]))
    d['reference_specimens'] = sorted(set(d['specimen'][~d['case']]))
    d['index'] = pd.Index(d['genes'])
    d['names'] = pd.Series({p: p.split('::', 1)[1] for p in d['sets']}).str.replace(r' R-HSA-\d+$', '', regex=True)


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only); raises when its cache was purged."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


def model_from(payload, genes):
    index = pd.Index(genes, name='gene')
    stats = pd.DataFrame({c: payload[f'stat|{c}'] for c in ('T_level', 'T_spatial', 'T_total', 'alpha', 'converged',
                                                            'separated')}, index=index)
    stats[['converged', 'separated']] = stats[['converged', 'separated']].astype(bool)
    names = sorted(k.split('|', 1)[1] for k in payload if k.startswith('curve|'))
    return GeneModel(statistics=stats, grid=payload['grid'], knots=payload['knots'], reference=payload['reference'],
                     case=payload['case'], delta=payload['delta'], se=payload['se'],
                     specimen_curves={n: payload[f'curve|{n}'] for n in names},
                     specimen_converged=pd.DataFrame({n: payload[f'converged|{n}'].astype(bool) for n in names}, index=index),
                     specimen_separated=pd.DataFrame({n: payload[f'separated|{n}'].astype(bool) for n in names}, index=index),
                     groups=pd.Series(payload['group_values'].astype(int), index=payload['group_names'].astype(str)),
                     dispersion={'alpha': payload['alpha'], 'converged': payload['alpha_converged'].astype(bool)})


MODELS, STAT71 = {ds: {} for ds in DATASETS}, {ds: {} for ds in DATASETS}
for ds, d in DATA.items():
    table = pd.read_csv(T71 / 'tables' / f'gene_statistics_{ds}.csv', header=[0, 1], index_col=0)
    assert table.index.astype(str).tolist() == d['genes']
    for label in PARTITIONS[ds]:
        model = model_from(stored(T71, f'gene_model_{ds}_{slug(label)}'), d['genes'])
        frame = table[SHORT[ds][label]].copy()
        frame[['converged', 'separated']] = frame[['converged', 'separated']].astype(bool)
        np.testing.assert_allclose(model.statistics.T_spatial.to_numpy(), frame.T_spatial.to_numpy(), rtol=1e-12, atol=0)
        df2 = DF2['condition' if label == 'species' else 'relabeling']
        again = moderated_f(frame.T_spatial, DF1, frame.LR_spec.to_numpy(), df2, covariate=d['log_mean_count'], robust=False)
        np.testing.assert_allclose(again.z.to_numpy(), frame.z.to_numpy(), rtol=1e-9, atol=1e-9)
        np.testing.assert_allclose(model.grid, common_grid(d['position'], d['specimen'], GRID_POINTS), rtol=0, atol=1e-12)
        MODELS[ds][label], STAT71[ds][label] = model, frame
    del d['counts']
guard_log.append("G1: the universe, sets and units equal notebook 71's; its six cached gene models reproduce its T_spatial and z")
print('\n'.join('· ' + g for g in guard_log))
print(f'Inputs {time.perf_counter() - t0:.0f} s')

### 1.2 · Segments and gene shapes

- **Cuts** c12 and c23: on the reference specimens' units, the positions that misassign the fewest units (S1 against
  S2 + S3; S1 + S2 against S3).
- **Gene shape** δ_g(x) = (Δ_g(x) − its grid mean) / ln 2 for usable genes (converged, not separated, finite Δ), its
  half-range, its extreme (up or down) and the extreme's segment; a gene is *late* when it is up in S3 or down in S1.
- **Specimen shapes** (A4): each case specimen's curve minus the mean of the two reference curves, centred, in log2,
  for genes converged and not separated in all four specimen fits.
- **Post hoc** (added after the development run showed that both case specimens share the reference-mean term):
  specimen pairs with no shared term, case k minus reference k (sorted names), for a replication sensitivity.

In [ ]:
from pseudospace import pathway_networks as pn

CUTS, SHAPE, GT, SPEC_SHAPES, PAIR_SHAPES = {}, {ds: {} for ds in DATASETS}, {ds: {} for ds in DATASETS}, {}, {}
CUT_ROWS = []
for ds, d in DATA.items():
    ref = ~d['case']
    CUTS[ds] = pn.segment_cuts(d['position'][ref], d['segment'][ref])
    grid = MODELS[ds]['species'].grid
    for label in PARTITIONS[ds]:
        frame = STAT71[ds][label]
        usable = (frame.converged & ~frame.separated).to_numpy()
        SHAPE[ds][label], GT[ds][label] = pn.gene_shapes(MODELS[ds][label].delta, usable, grid, CUTS[ds])
    model = MODELS[ds]['species']
    ok = (model.specimen_converged.all(axis=1) & ~model.specimen_separated.any(axis=1)).to_numpy()
    reference = np.mean([model.specimen_curves[n] for n in d['reference_specimens']], axis=0)
    shapes = []
    for n in d['case_specimens']:
        diff = model.specimen_curves[n] - reference
        good = ok & np.isfinite(diff).all(axis=1)
        s = np.full(diff.shape, np.nan)
        s[good] = (diff[good] - diff[good].mean(axis=1, keepdims=True)) / np.log(2)
        shapes.append(s)
    SPEC_SHAPES[ds] = tuple(shapes)
    pairs = []                                                    # post hoc: case k against reference k (no shared term)
    for n, r in zip(d['case_specimens'], d['reference_specimens']):
        diff = model.specimen_curves[n] - model.specimen_curves[r]
        good = ok & np.isfinite(diff).all(axis=1)
        s = np.full(diff.shape, np.nan)
        s[good] = (diff[good] - diff[good].mean(axis=1, keepdims=True)) / np.log(2)
        pairs.append(s)
    PAIR_SHAPES[ds] = tuple(pairs)
    seg = pn.grid_segments(grid, CUTS[ds])
    CUT_ROWS.append({'dataset': d['title'], 'c12': CUTS[ds][0], 'c23': CUTS[ds][1], 'grid start': grid[0], 'grid end': grid[-1],
                     **{f'grid points in {s}': int((seg == s).sum()) for s in ('S1', 'S2', 'S3')},
                     'usable genes (condition)': int(GT[ds]['species'].usable.sum()),
                     'genes usable in all four specimen fits': int(np.isfinite(shapes[0]).all(axis=1).sum())})
CUT_TABLE = pd.DataFrame(CUT_ROWS).set_index('dataset')
CUT_TABLE.to_csv(tables / 'segment_cuts.csv')
display(CUT_TABLE.round(3))

## 2 · The three methods' calls and driver genes

- **F** (71's positional F-test, cameraPR Up and FDR ≤ 0.05): driver genes are usable members with gene q ≤ 0.05;
  fewer than 3 → usable members in the top 10% of z (75's contributing genes); still fewer → all usable members.
- **C** (71's clustering, default null; the relabelings under the fair null): the union of the overlap genes over
  the lists where the set has matched q < 0.05.
- **N** (row N): genes usable, q ≤ 0.05 and half-range ≥ 0.5 log2, in six lists (up or down × the extreme's
  segment); ORA (`family_ora`) of the sets with 10–300 members among the usable genes, background = the usable
  genes, BH over set × list; call = any list q ≤ 0.05; calibration p = min(1, 6 × best p). Driver genes: the union of
  members in the lists where the set has q ≤ 0.05.

**Guards:** G2, F's and C's calls and relabeled calls reproduce 71's method table; G5, the clustering ORA recomputed
from the condition run's gene groups reproduces its calls (71's G7).

In [ ]:
from scipy.stats import norm

from pseudospace.discrete_comparators import family_ora, membership, swap_rule
from pseudospace_reconstruction.compare.pathways import PathwayParams, ora_by_group
from pseudospace_reconstruction.io.gene_sets import build_gene_sets

PATHWAY_PARAMS = PathwayParams(min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])
ORA_KW = dict(n_strata=PATHWAY_PARAMS.n_strata, min_set_size=PATHWAY_PARAMS.min_genes,
              min_group_size=PATHWAY_PARAMS.min_group_size, min_overlap=PATHWAY_PARAMS.min_overlap)
EXPRESSION_COLUMN = 'expression_per_10k'


def unflatten(payload):
    names = sorted({k.split('|', 1)[0] for k in payload if not k.startswith('__')})
    return {n: pd.DataFrame({c: payload[f'{n}|{c}'] for c in payload[f'{n}|__columns'].astype(str)}) for n in names}


def b_calls(t):
    return set(t['ora'][t['ora'].q_matched.astype(float) < ALPHA].pathway_id)


def b_bonferroni_p(sets, t):
    """75's list-based convention: the best matched ORA p over the lists x the number of lists."""
    genes = t['genes']
    n_lists = sum(int((genes[c][genes[c].ne('')].value_counts() >= PATHWAY_PARAMS.min_group_size).sum()) for c in ('group', 'cluster'))
    best = t['ora'].groupby('pathway_id').p_matched.min().astype(float)
    return (best * max(n_lists, 1)).clip(upper=1.).reindex(list(sets)).fillna(1.).to_numpy()


def b_drivers(t):
    ora = t['ora'][t['ora'].q_matched.astype(float) < ALPHA]
    out = {}
    for p, block in ora.groupby('pathway_id'):
        out[p] = set(g for s in block.overlap_genes.astype(str) for g in s.split(',') if g)
    return out


def clustering_calls(genes_table, sets, universe=None):
    """The clustering method's ORA calls from a run's gene groups and clusters (the package's ora_by_group; 71's code)."""
    expression = genes_table[EXPRESSION_COLUMN].to_numpy(float)
    ora = pd.concat([ora_by_group(genes_table[k].to_numpy(), sets, expression, universe=universe, **ORA_KW).assign(list=k)
                     for k in ('group', 'cluster')], ignore_index=True)
    return set((ora.library + '::' + ora.pathway)[ora.q_matched < ALPHA])


def contributing_genes(z):
    """75's contributing genes: the top 10% of all genes by z."""
    z = pd.Series(z)
    return set(z.nlargest(int(round(TOP_SHARE * len(z)))).index)


def f_drivers(ds, calls, frame, table, sets):
    """F's driver genes: usable members with q <= 0.05; then usable members among the top 10% z; then all usable."""
    usable = set(frame.index[table.usable.to_numpy()])
    significant = usable & set(frame.index[frame.q.le(ALPHA).to_numpy()])
    top = usable & contributing_genes(frame.z)
    out = {}
    for p in calls:
        members = set(sets[p])
        chosen = members & significant
        if len(chosen) < MIN_DRIVERS:
            chosen = members & top
        if len(chosen) < MIN_DRIVERS:
            chosen = members & usable
        out[p] = chosen
    return out


def row_n(genes, sets, table, q, *, keep=None):
    """Row N on one partition. Returns (per-set DataFrame over tested sets, {call: driver genes}, tested set ids)."""
    usable = table.usable.to_numpy() & (np.ones(len(genes), bool) if keep is None else np.asarray(keep, bool))
    selected = usable & (np.asarray(q, float) <= ALPHA) & (table.half_range.to_numpy() >= HALF_RANGE)
    lists = pn.position_split_lists(table.assign(usable=usable), selected)
    background = np.flatnonzero(usable)
    bg = [genes[i] for i in background]
    restricted = {p: [g for g in v if g in set(bg)] for p, v in sets.items()} if keep is not None else sets
    member = membership(restricted, bg)
    size = np.asarray(member.sum(axis=1)).ravel()
    tested = (size >= SET_SIZES[0]) & (size <= SET_SIZES[1])
    ids = [p for p, t in zip(sets, tested) if t]
    ora = family_ora([lists[k][background] for k in LIST_KEYS], member[np.flatnonzero(tested)])
    ora.index = pd.Index(ids, name='pathway_id')
    ora = ora.rename(columns={**{f'p_{i}': f'p {d} {s}' for i, (d, s) in enumerate(LIST_KEYS)},
                              **{f'q_{i}': f'q {d} {s}' for i, (d, s) in enumerate(LIST_KEYS)}})
    list_genes = {k: set(np.asarray(genes)[lists[k]]) for k in LIST_KEYS}
    drivers = {}
    for p in ora.index[ora.call]:
        drivers[p] = set().union(*[set(sets[p]) & list_genes[k] for k in LIST_KEYS if ora.loc[p, f'q {k[0]} {k[1]}'] <= ALPHA])
    ora['selected genes'] = int(selected.sum())
    return ora, drivers, ids


EXT_SETS, B, PRIMARY = {}, {ds: {} for ds in DATASETS}, {}
CALLS = {ds: {m: {} for m in METHODS} for ds in DATASETS}       # calls[ds][method][label] = set
DRIVERS = {ds: {m: {} for m in METHODS} for ds in DATASETS}     # drivers[ds][method][label] = {call: set of genes}
BESTP = {ds: {m: {} for m in METHODS} for ds in DATASETS}       # best p per call, for ordering programs
CALIB = {ds: {} for ds in DATASETS}
TESTED = {ds: {} for ds in DATASETS}
ROW_N = {ds: {} for ds in DATASETS}
t0 = time.perf_counter()
for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    condition, r1, r2 = labels
    sets = build_gene_sets({lib: {p.split('::', 1)[1]: v for p, v in d['sets'].items() if p.startswith(lib + '::')}
                            for lib in LIBRARIES}, d['genes'], min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])
    EXT_SETS[ds] = sets
    # F
    PRIMARY[ds] = {l: pd.read_csv(T71 / 'tables' / f'pathway_tests_{ds}_cameraPR_{slug(l)}.csv').set_index('pathway_id') for l in labels}
    for l in labels:
        t = PRIMARY[ds][l]
        CALLS[ds]['F'][l] = set(t.index[t.call.astype(bool)])
        DRIVERS[ds]['F'][l] = f_drivers(ds, CALLS[ds]['F'][l], STAT71[ds][l], GT[ds][l], d['sets'])
        BESTP[ds]['F'][l] = t.p_one_sided.astype(float)
    CALIB[ds]['F'] = np.concatenate([PRIMARY[ds][l].p_one_sided.to_numpy(float) for l in (r1, r2)])
    TESTED[ds]['F'] = len(d['sets'])
    # C
    runs = {condition: (condition, None), r1: (r1, r2), r2: (r2, r1)}
    for l, (contrast, null) in runs.items():
        stage = f'method_b_{ds}_{slug(contrast)}__null_{"default" if null is None else slug(null)}'
        B[ds][l] = unflatten(stored(T71, stage))
        CALLS[ds]['C'][l] = b_calls(B[ds][l])
        DRIVERS[ds]['C'][l] = b_drivers(B[ds][l])
        BESTP[ds]['C'][l] = B[ds][l]['ora'].groupby('pathway_id').p_matched.min().astype(float)
        assert set(DRIVERS[ds]['C'][l]) == CALLS[ds]['C'][l]
    CALIB[ds]['C'] = np.concatenate([b_bonferroni_p(d['sets'], B[ds][l]) for l in (r1, r2)])
    TESTED[ds]['C'] = len(d['sets'])
    genes_table = B[ds][condition]['genes']
    assert list(genes_table.gene.astype(str)) == d['genes']
    assert clustering_calls(genes_table, EXT_SETS[ds]) == CALLS[ds]['C'][condition], 'G5: the clustering ORA does not reproduce its run'
    # N
    calib = []
    for l in labels:
        ora, drivers, ids = row_n(d['genes'], d['sets'], GT[ds][l], STAT71[ds][l].q.to_numpy())
        ROW_N[ds][l] = ora
        CALLS[ds]['N'][l] = set(ora.index[ora.call])
        DRIVERS[ds]['N'][l] = drivers
        BESTP[ds]['N'][l] = ora.best_p
        if l == condition:
            TESTED[ds]['N'] = len(ids)
        else:
            calib.append(ora.calibration_p.to_numpy(float))
    CALIB[ds]['N'] = np.concatenate(calib)
    pd.concat({SHORT[ds][l]: ROW_N[ds][l] for l in labels}, axis=1).to_csv(tables / f'row_n_{ds}.csv')
    gt = GT[ds][condition]
    pd.DataFrame({'gene': d['genes'], 'usable': gt.usable, 'q': STAT71[ds][condition].q.to_numpy(), 'z': STAT71[ds][condition].z.to_numpy(),
                  'half_range': gt.half_range, 'direction': gt.direction, 'segment': gt.segment, 'side': gt.side,
                  'selected (row N)': gt.usable & STAT71[ds][condition].q.le(ALPHA).to_numpy() & (gt.half_range >= HALF_RANGE)}
                 ).to_csv(tables / f'row_n_genes_{ds}.csv', index=False)
guard_log.append('G5: the clustering ORA recomputed from the condition runs\' gene groups reproduces their calls, both datasets')

# G2: F's and C's calls and relabeled calls reproduce 71's method table
M71 = pd.read_csv(T71 / 'tables' / 'method_table.csv')
for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    for m, method in (('F', 'normalized · cameraPR on z'), ('C', 'clustering · ORA, default null')):
        theirs = M71[M71.dataset.eq(d['title']) & M71.method.eq(method)].iloc[0]
        assert str(len(CALLS[ds][m][labels[0]])) == str(theirs['calls']), (ds, m)
        assert ', '.join(str(len(CALLS[ds][m][l])) for l in labels[1:]) == theirs['relabeled calls'], (ds, m)
guard_log.append("G2: F's and C's calls and relabeled calls reproduce notebook 71's method table")
print('\n'.join('· ' + g for g in guard_log[-2:]))
CALL_COUNTS = pd.DataFrame([{'dataset': DATA[ds]['title'], 'method': METHODS[m], **{SHORT[ds][l]: len(CALLS[ds][m][l]) for l in PARTITIONS[ds]},
                             'tested': TESTED[ds][m],
                             'selected genes (condition)': int(ROW_N[ds]['species']['selected genes'].iloc[0]) if m == 'N' else np.nan}
                            for ds in DATASETS for m in METHODS])
display(CALL_COUNTS)
print(f'Calls {time.perf_counter() - t0:.0f} s')

## 3 · Q1: calling criteria (F, C and N)

- **V1 · swap rule** (75's).
- **V2 · reconstruction stability:** the Jaccard of the condition calls of notebooks 69 and 71 over the sets tested in
  both runs. Row N is rerun on 69's gene models, gene q and sets, with 69's cuts (69's positions and labels of the
  reference units present in both runs). **G4:** 69's sets equal its saved tables, and F's and C's Jaccard values
  reproduce 71's `compare69_pathways_summary.csv`.
- **V3 · probe exposure** (human vs mouse): row N rerun on the equal-probe genes; F's and C's values from 75.
- **V4 · hub-gene robustness:** the share of calls kept without the method's 10 hub genes (driver genes of the most
  calls; ties to higher condition z).
- Descriptive: calls in redundancy clusters, offset lean (75's rule), overlap of N with F and C.

In [ ]:
from pseudospace.coordinate_inputs import pt_inputs, read_coordinate
from pseudospace.pathways import build_pathway_membership

t0 = time.perf_counter()
GENES69 = {ds: pd.read_csv(T69 / 'tables' / f'gene_statistics_{ds}.csv', header=[0, 1], index_col=0).index.astype(str).tolist()
           for ds in DATASETS}


def pathway_sets(universe, tested=None):
    """Notebook 37's pathways: members among the universe, 10-300 of them tested (notebook 69's helper, 71's code)."""
    tested = list(universe) if tested is None else tested
    coverage = pd.concat([build_pathway_membership(
        json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json').read_text()), universe,
        ortholog_map=orthologs, library_name=lib, tested=tested, min_genes=1) for lib in LIBRARIES], ignore_index=True)
    coverage['pathway_id'] = coverage.library + '::' + coverage.pathway
    coverage = coverage[coverage.n_tested.between(SET_SIZES[0], min(SET_SIZES[1], len(tested) - 1))]
    return {row.pathway_id: list(row.genes_present) for row in coverage.itertuples()}


def restrict(sets, genes):
    keep = set(genes)
    sets = {p: [g for g in v if g in keep] for p, v in sets.items()}
    return {p: v for p, v in sets.items() if SET_SIZES[0] <= len(v) <= min(SET_SIZES[1], len(keep) - 1)}


nb12 = [g for g in pd.read_csv(nb12_universe_file, index_col=0).index.astype(str) if g not in MT]
inputs69 = pt_inputs(results_root, data_root, read_coordinate(coordinate_file), exclude_genes=MT)
assert list(inputs69['genes']) == GENES69[hvm]
SETS69 = {hvm: inputs69['gene_sets'], aki: restrict(pathway_sets(nb12), GENES69[aki])}
del inputs69
TESTS69 = {ds: pd.read_csv(T69 / 'tables' / f'pathway_tests_{ds}_cameraPR_species.csv').set_index('pathway_id') for ds in DATASETS}
RED69 = {ds: pd.read_csv(T69 / 'tables' / f'redundancy_clusters_{ds}.csv', index_col='pathway_id') for ds in DATASETS}
for ds in DATASETS:                                                                                          # G4 (sets)
    assert set(SETS69[ds]) == set(TESTS69[ds].index) == set(RED69[ds].index), ds
    assert all(len(SETS69[ds][p]) == int(TESTS69[ds].loc[p, 'n']) for p in SETS69[ds]), ds

R69 = {}
for ds, d in DATA.items():
    g69 = pd.read_csv(T69 / 'tables' / f'gene_statistics_{ds}.csv', header=[0, 1], index_col=0)['condition']
    g69[['converged', 'separated']] = g69[['converged', 'separated']].astype(bool)
    model69 = model_from(stored(T69, f'gene_model_{ds}_species'), GENES69[ds])
    um = pd.read_csv(T71 / 'tables' / f'unit_matching_{ds}.csv')
    ref = um[um['sample'].isin(d['reference_specimens']) & um.in_set_69.astype(bool)].dropna(subset=['old_position_69'])
    cuts69 = pn.segment_cuts(ref.old_position_69.to_numpy(float), ref.old_segment_69.astype(str).str.replace('PT-', '', regex=False).to_numpy())
    shape69, gt69 = pn.gene_shapes(model69.delta, (g69.converged & ~g69.separated).to_numpy(), model69.grid, cuts69)
    ora69, drivers69, ids69 = row_n(GENES69[ds], SETS69[ds], gt69, g69.q.to_numpy())
    b69 = unflatten(stored(T69, f'method_b_{ds}_species__null_default'))
    calls69 = {'F': set(TESTS69[ds].index[TESTS69[ds].call.astype(bool)]), 'C': b_calls(b69), 'N': set(ora69.index[ora69.call])}
    drivers = {'F': f_drivers(ds, calls69['F'], g69, gt69, SETS69[ds]), 'C': b_drivers(b69), 'N': drivers69}
    bestp = {'F': TESTS69[ds].p_one_sided.astype(float), 'C': b69['ora'].groupby('pathway_id').p_matched.min().astype(float),
             'N': ora69.best_p}
    R69[ds] = {'calls': calls69, 'drivers': drivers, 'bestp': bestp, 'shape': shape69, 'gt': gt69, 'cuts': cuts69,
               'z': g69.z, 'genes': GENES69[ds], 'tested_both': set(d['sets']) & set(SETS69[ds]), 'grid': model69.grid}

SUMMARY69 = pd.read_csv(T71 / 'tables' / 'compare69_pathways_summary.csv')
CRIT = {ds: {m: {} for m in METHODS} for ds in DATASETS}


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else np.nan


for ds, d in DATA.items():
    both = R69[ds]['tested_both']
    for m in METHODS:
        CRIT[ds][m]['reconstruction Jaccard'] = jaccard(R69[ds]['calls'][m] & both, CALLS[ds][m]['species'] & both)
    for m, method in (('F', 'normalized'), ('C', 'clustering')):                                            # G4 (Jaccard)
        theirs = SUMMARY69[SUMMARY69.dataset.eq(d['title']) & SUMMARY69.method.eq(method)].iloc[0]
        assert np.isclose(CRIT[ds][m]['reconstruction Jaccard'], theirs['Jaccard (pathways)']), (ds, m)
guard_log.append("G4: notebook 69's sets equal its saved tables; F's and C's 69 -> 71 Jaccard reproduce notebook 71's")
print(guard_log[-1])
print(f'Notebook 69 rebuilt {time.perf_counter() - t0:.0f} s')

In [ ]:
t0 = time.perf_counter()
REDUNDANCY = {ds: pd.read_csv(T71 / 'tables' / f'redundancy_clusters_{ds}.csv', index_col=0).reindex(list(DATA[ds]['sets']))
              for ds in DATASETS}
PROBE = pd.read_csv(nb38 / 'probe_counts.csv')
EXPOSURE75 = pd.read_csv(T75 / 'tables' / 'probe_exposure.csv').set_index('row')
HUBS, HUB_ROWS, LEAN = {}, [], {}


def without(sets, drop):
    out = {p: [g for g in v if g not in drop] for p, v in sets.items()}
    return {p: v for p, v in out.items() if len(v) >= SET_SIZES[0]}


for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    condition = labels[0]
    z = STAT71[ds][condition].z
    # V1
    for m in METHODS:
        verdict = swap_rule([len(CALLS[ds][m][condition])], [len(CALLS[ds][m][l]) for l in labels[1:]], TESTED[ds][m], CALIB[ds][m])
        CRIT[ds][m].update({'swap rule': 1. if verdict['pass'] == 'pass' else 0., 'NCDR': verdict['NCDR'],
                            'calibration share': verdict['calibration share'], 'swap verdict': verdict['pass'],
                            'relabeling rule': verdict['relabeling rule'], 'calibration': verdict['calibration']})
    # V3
    if ds == hvm:
        balanced = set(PROBE.mouse_symbol[PROBE.probe_balance.eq('balanced')].astype(str)) & set(d['genes'])
        ora_eq, _, _ = row_n(d['genes'], d['sets'], GT[ds][condition], STAT71[ds][condition].q.to_numpy(),
                             keep=np.isin(d['genes'], list(balanced)))
        calls_n, calls_eq = CALLS[ds]['N'][condition], set(ora_eq.index[ora_eq.call])
        CRIT[ds]['N']['probe exposure'] = 1 - len(calls_n & calls_eq) / len(calls_n) if calls_n else np.nan
        CRIT[ds]['N']['equal-probe calls'] = len(calls_eq)
        CRIT[ds]['F']['probe exposure'] = float(EXPOSURE75.loc['primary', 'exposure'])
        CRIT[ds]['C']['probe exposure'] = float(EXPOSURE75.loc['clustering', 'exposure'])
    # V4
    for m in METHODS:
        counts = Counter(g for genes in DRIVERS[ds][m][condition].values() for g in genes)
        ranked = sorted(counts, key=lambda g: (-counts[g], -float(z.get(g, -np.inf)), g))
        hubs = ranked[:N_HUB]
        HUBS[(ds, m)] = hubs
        calls = CALLS[ds][m][condition]
        if m == 'F':
            camera = camera_pr(z.drop(hubs), without(d['sets'], set(hubs)), inter_gene_cor=CAMERA_COR, fdr=ALPHA)
            after = set(camera.pathway_id[camera.call])
        elif m == 'C':
            after = clustering_calls(B[ds][condition]['genes'], EXT_SETS[ds], universe=~np.isin(d['genes'], hubs))
        else:
            ora_h, _, _ = row_n(d['genes'], d['sets'], GT[ds][condition], STAT71[ds][condition].q.to_numpy(), keep=~np.isin(d['genes'], hubs))
            after = set(ora_h.index[ora_h.call])
        CRIT[ds][m]['hub robustness'] = len(calls & after) / len(calls) if calls else np.nan
        CRIT[ds][m]['calls after hub removal'] = len(after)
        HUB_ROWS.extend({'dataset': d['title'], 'method': METHODS[m], 'rank': i + 1, 'gene': g, 'calls holding it': counts[g],
                         'share of calls': counts[g] / len(calls), 'z': float(z.get(g, np.nan))} for i, g in enumerate(hubs))
    # descriptive
    s1 = pd.read_csv(T71 / 'tables' / f's1_pathway_tests_{ds}.csv', header=[0, 1], index_col=0)
    level_p = s1['level · cameraPR · condition'].p_two_sided.astype(float).clip(1e-300, 1)
    shape_p = s1['shape · cameraPR · condition'].p.astype(float).clip(1e-300, 1 - 1e-16)
    LEAN[ds] = pd.Series(norm.isf(level_p / 2) > norm.isf(shape_p), index=s1.index)
    for m in METHODS:
        calls = CALLS[ds][m][condition]
        CRIT[ds][m]['calls'] = len(calls)
        CRIT[ds][m]['calls (redundancy clusters)'] = len(set(REDUNDANCY[ds].representative.reindex(sorted(calls))))
        CRIT[ds][m]['offset-leaning share'] = float(LEAN[ds].reindex(sorted(calls)).mean()) if calls else np.nan
    for other in ('F', 'C'):
        a, b = CALLS[ds]['N'][condition], CALLS[ds][other][condition]
        ra, rb = set(REDUNDANCY[ds].representative.reindex(sorted(a))), set(REDUNDANCY[ds].representative.reindex(sorted(b)))
        CRIT[ds]['N'][f'Jaccard with {other} (sets)'] = jaccard(a, b)
        CRIT[ds]['N'][f'Jaccard with {other} (redundancy clusters)'] = jaccard(ra, rb)
        CRIT[ds]['N'][f'both with {other}'] = len(a & b)
        CRIT[ds]['N'][f'N only vs {other}'] = len(a - b)
        CRIT[ds]['N'][f'{other} only'] = len(b - a)
pd.DataFrame(HUB_ROWS).to_csv(tables / 'hub_genes.csv', index=False)
CALLING = pd.DataFrame([{'dataset': DATA[ds]['title'], 'method': METHODS[m], **CRIT[ds][m]} for ds in DATASETS for m in METHODS])
CALLING.to_csv(tables / 'calling_criteria.csv', index=False)
VERDICT_ROWS = []
for ds in DATASETS:
    for other in ('F', 'C'):
        a = {k: CRIT[ds]['N'].get(k, np.nan) for k in CALL_DIRECTIONS}
        b = {k: CRIT[ds][other].get(k, np.nan) for k in CALL_DIRECTIONS}
        VERDICT_ROWS.append({'dataset': DATA[ds]['title'], 'comparison': f'N against {other}',
                             'verdict': pn.dominance(a, b, CALL_DIRECTIONS, CALL_TOL),
                             **{f'{k} (N; {other})': f'{a[k]:.3f}; {b[k]:.3f}' for k in CALL_DIRECTIONS if np.isfinite(a[k]) or np.isfinite(b[k])}})
CALL_VERDICTS = pd.DataFrame(VERDICT_ROWS)
CALL_VERDICTS.to_csv(tables / 'calling_verdicts.csv', index=False)
display(CALLING.set_index(['dataset', 'method'])[['calls', 'calls (redundancy clusters)', 'swap verdict', 'NCDR', 'calibration share',
                                                  'reconstruction Jaccard', 'probe exposure', 'hub robustness',
                                                  'offset-leaning share']].round(3))
display(CALL_VERDICTS)
print(f'Q1 {time.perf_counter() - t0:.0f} s')

## 4 · The networks, names and positional attributes

- **NET:** nodes are a method's calls; an edge joins two calls whose driver genes have Jaccard ≥ 0.1, weighted by it;
  Leiden (modularity, resolution 1, β 0.01, iterated until stable, seed 1234). Isolates are one-call programs.
- **NET-pos:** an edge joins two calls whose call profiles correlate at r ≥ 0.8, weighted by r; same Leiden.
- **OVL:** 75's overlap programs; **RED:** 71's redundancy clusters, restricted to the calls.
- **Call attributes:** profile (mean δ over usable driver genes), peak and trough, half-range, position score
  (n_late − n_early)/√n with its class.
- **Names:** scoranet's ancestor rule on Reactome and KEGG; otherwise the hub call.

**Guards:** G3, OVL on F's calls reproduces 75's programs; G7, Leiden run twice with the same seed gives the same
membership.

In [ ]:
from sklearn.metrics import adjusted_rand_score

from pseudospace.pathway_decoys import overlap_programs

REACTOME = pn.reactome_hierarchy(HIER / 'ReactomePathwaysRelation.txt', HIER / 'ReactomePathways.txt')
KEGG, KEGG_LOOKUP = pn.kegg_hierarchy(HIER / 'br08901.keg')
HIERARCHIES = {'R-HSA-': REACTOME, 'K:': KEGG}


def hierarchy_node(pathway_id):
    library, name = pathway_id.split('::', 1)
    if library == 'Reactome_2022':
        m = pd.Series([name]).str.extract(r'(R-HSA-\d+)$')[0].iloc[0]
        return m if isinstance(m, str) and m in REACTOME.names else None
    if library == 'KEGG_2019_Mouse':
        return pn.kegg_node(name, KEGG_LOOKUP)
    return None


NODES = {p: hierarchy_node(p) for ds in DATASETS for p in DATA[ds]['sets']}
NODE_ROWS = [{'library': lib, 'sets': sum(p.startswith(lib) for p in NODES),
              'with a hierarchy node': sum(p.startswith(lib) and NODES[p] is not None for p in NODES)} for lib in LIBRARIES]
display(pd.DataFrame(NODE_ROWS))


def node_name(node):
    return REACTOME.names.get(node) or KEGG.names.get(node) or node


def call_profiles(shape, drivers, calls, index):
    """Calls x grid: mean delta over the usable driver genes (0 where a call has none)."""
    out = np.zeros((len(calls), shape.shape[1]))
    for i, c in enumerate(calls):
        rows = index.get_indexer(sorted(drivers[c]))
        rows = rows[rows >= 0]
        rows = rows[np.isfinite(shape[rows]).all(axis=1)]
        if len(rows):
            out[i] = shape[rows].mean(axis=0)
    return out


def build_groupings(sets, calls, drivers, bestp, shape, index, z, red, *, threshold=JACCARD_EDGE, resolution=RESOLUTION,
                    cut=PROGRAM_CUT, only=None):
    """{grouping: Series call -> program} on one partition's calls."""
    calls = sorted(calls)
    out = {}
    if not calls:
        return {g: pd.Series(dtype=int) for g in GROUPINGS}
    p = pd.Series(bestp).reindex(calls).fillna(1.).to_numpy()
    if only in (None, 'NET'):
        names, J = pn.driver_jaccard({c: drivers[c] for c in calls})
        out['NET'] = pn.leiden_programs(names, J, threshold=threshold, resolution=resolution, seed=LEIDEN_SEED, best_p=p)
    if only in (None, 'NET-pos'):
        R = pn.profile_correlation(call_profiles(shape, drivers, calls, index))
        out['NET-pos'] = pn.leiden_programs(calls, R, threshold=PROFILE_EDGE, resolution=RESOLUTION, seed=LEIDEN_SEED, best_p=p)
    if only in (None, 'OVL'):
        out['OVL'] = overlap_programs(sets, calls, contributing=contributing_genes(z), cut=cut).set_index('pathway_id').program
    if only in (None, 'RED'):
        out['RED'] = pd.Series(pd.factorize(pd.Series(red).reindex(calls).to_numpy())[0] + 1, index=calls, name='program')
    return out


GROUPS = {ds: {m: {} for m in METHODS} for ds in DATASETS}      # GROUPS[ds][m][label][grouping]
for ds, d in DATA.items():
    for m in METHODS:
        for l in PARTITIONS[ds]:
            GROUPS[ds][m][l] = build_groupings(d['sets'], CALLS[ds][m][l], DRIVERS[ds][m][l], BESTP[ds][m][l], SHAPE[ds][l], d['index'],
                                               STAT71[ds][l].z, REDUNDANCY[ds].representative)
# G3: OVL on F's calls reproduces 75's programs (same partition)
for ds in DATASETS:
    theirs = pd.read_csv(T75 / 'tables' / f'programs_{ds}.csv', index_col=0).program
    mine = GROUPS[ds]['F']['species']['OVL']
    assert set(theirs.index) == set(mine.index) and adjusted_rand_score(theirs.reindex(mine.index), mine) == 1., ds
# G7: Leiden determinism
for ds, d in DATA.items():
    again = build_groupings(d['sets'], CALLS[ds]['C']['species'], DRIVERS[ds]['C']['species'], BESTP[ds]['C']['species'],
                            SHAPE[ds]['species'], d['index'], STAT71[ds]['species'].z, REDUNDANCY[ds].representative, only='NET')['NET']
    assert again.equals(GROUPS[ds]['C']['species']['NET']), ds
guard_log.extend(["G3: the overlap programs on F's calls reproduce notebook 75's programs, both datasets",
                  'G7: Leiden with the same seed gives the same membership'])
print('\n'.join('· ' + g for g in guard_log[-2:]))


def program_names(calls_by_program, bestp, degree=None):
    """{program: (name, named by an ancestor)}: scoranet's ancestor rule, else the hub call."""
    out = {}
    for prog, calls in calls_by_program.items():
        picks, _ = pn.name_program([NODES.get(c) for c in calls], HIERARCHIES, min_share=MIN_SHARE,
                                   max_descendant_share=MAX_DESC_SHARE, max_picks=MAX_PICKS)
        if picks:
            out[prog] = (' · '.join(node_name(x) for x in picks), True)
        else:
            key = (lambda c: (-(degree or {}).get(c, 0), float(bestp.get(c, 1.)), c))
            hub = sorted(calls, key=key)[0]
            out[prog] = (hub.split('::', 1)[1].replace(' R-HSA-', ' (R-HSA-') + (')' if 'R-HSA-' in hub else ''), False)
    return out


def network_degree(calls, drivers):
    names, J = pn.driver_jaccard({c: drivers[c] for c in sorted(calls)})
    A = (J >= JACCARD_EDGE) & ~np.eye(len(names), dtype=bool)
    return dict(zip(names, A.sum(axis=1)))


CALL_TABLES, NAMES = {}, {ds: {m: {} for m in METHODS} for ds in DATASETS}
for ds, d in DATA.items():
    grid = MODELS[ds]['species'].grid
    seg = pn.grid_segments(grid, CUTS[ds])
    rows = []
    for m in METHODS:
        calls = sorted(CALLS[ds][m]['species'])
        drivers = DRIVERS[ds][m]['species']
        profiles = call_profiles(SHAPE[ds]['species'], drivers, calls, d['index'])
        degree = network_degree(calls, drivers) if calls else {}
        bestp = pd.Series(BESTP[ds][m]['species'])
        for g, labels_ in GROUPS[ds][m]['species'].items():
            by_prog = {k: list(v.index) for k, v in labels_.groupby(labels_)}
            NAMES[ds][m][g] = program_names(by_prog, bestp, degree if g == 'NET' else None)
        side = GT[ds]['species'].side
        for i, c in enumerate(calls):
            rows_ = d['index'].get_indexer(sorted(drivers[c]))
            rows_ = rows_[(rows_ >= 0)]
            usable = rows_[GT[ds]['species'].usable.to_numpy()[rows_]]
            score = pn.position_score(side.to_numpy()[usable])
            prof = profiles[i]
            rows.append({'method': METHODS[m], 'pathway_id': c, 'pathway': d['names'][c], 'best p': float(bestp.get(c, np.nan)),
                         'driver genes': len(drivers[c]), 'usable driver genes': len(usable),
                         'peak position': float(grid[np.argmax(prof)]), 'peak segment': seg[np.argmax(prof)],
                         'trough position': float(grid[np.argmin(prof)]), 'trough segment': seg[np.argmin(prof)],
                         'half-range (log2)': float((prof.max() - prof.min()) / 2), 'position score': score,
                         'position class': pn.score_class(score, SCORE_THRESHOLD), 'degree (NET)': int(degree.get(c, 0)),
                         **{f'program {g}': int(GROUPS[ds][m]['species'][g][c]) for g in GROUPINGS},
                         'program name (NET)': NAMES[ds][m]['NET'][GROUPS[ds][m]['species']['NET'][c]][0],
                         'driver gene list': ','.join(sorted(drivers[c]))})
    CALL_TABLES[ds] = pd.DataFrame(rows)
    CALL_TABLES[ds].to_csv(tables / f'calls_{ds}.csv', index=False)
display(pd.DataFrame([{'dataset': DATA[ds]['title'], 'method': METHODS[m], 'calls': len(CALLS[ds][m]['species']),
                       **{f'{g} programs': int(GROUPS[ds][m]['species'][g].nunique()) if len(GROUPS[ds][m]['species'][g]) else 0 for g in GROUPINGS}}
                      for ds in DATASETS for m in METHODS]))

## 5 · Q2: groupings on the same calls

- **W1** programs per call; **W2** coherence z and **W3** replication z (call-weighted means against 1,000 random
  partitions of the same calls with the grouping's own program sizes, seed 93); **W4** the adjusted Rand index between
  the grouping on notebook 69's calls and statistics and on 71's, over the calls in both; **W5** program NCDR (mean
  relabeled programs ÷ condition programs).
- W2 is not judged for NET-pos (it groups by profile similarity by construction).
- **Post hoc, not judged:** replication on independent specimen pairs (case k − reference k), with its own null.
- Settings sensitivity (descriptive): NET over threshold × resolution, OVL over cut, as ARI against the default.

In [ ]:
from pseudospace.stage_cache import cached_payload, digest

cache = output / 'stage_cache'
code_key = digest([NOTEBOOK_LOGIC_VERSION, Path(pn.__file__)])
SECONDS = {}


def cached(stage, compute, *, params=None, inputs=None):
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs=inputs, code=code_key)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


t0 = time.perf_counter()
STATS, PROGRAM_TABLES, GROUP_ROWS = {}, {}, []
for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    for m in METHODS:
        calls = sorted(CALLS[ds][m]['species'])
        if not calls:
            continue
        drivers = {c: [int(i) for i in d['index'].get_indexer(sorted(DRIVERS[ds][m]['species'][c])) if i >= 0] for c in calls}
        stats = pn.ProgramStats(shape=SHAPE[ds]['species'], drivers=drivers, specimen_shapes=SPEC_SHAPES[ds])
        STATS[(ds, m)] = stats
        pair_stats = pn.ProgramStats(shape=SHAPE[ds]['species'], drivers=drivers, specimen_shapes=PAIR_SHAPES[ds])
        r69 = R69[ds]
        groups69 = build_groupings(SETS69[ds], r69['calls'][m], r69['drivers'][m], r69['bestp'][m], r69['shape'], pd.Index(r69['genes']),
                                   r69['z'], RED69[ds].representative)
        for g in GROUPINGS:
            labels_ = GROUPS[ds][m]['species'][g]
            table, profiles = stats.per_program(labels_)
            coh, rep = stats.summary(labels_)
            null = cached(f'null_{ds}_{m}_{g}', lambda: {'null': stats.null(labels_, n=N_NULL, seed=NULL_SEED)},
                          params={'n': N_NULL, 'seed': NULL_SEED}, inputs={'labels': labels_.to_numpy(), 'calls': calls,
                                                                           'drivers': digest(json.dumps(drivers, sort_keys=True))})['null']
            shared = sorted(set(labels_.index) & set(groups69[g].index))
            rep_pair = pair_stats.summary(labels_)[1]
            null_pair = cached(f'null_pairs_{ds}_{m}_{g}', lambda: {'null': pair_stats.null(labels_, n=N_NULL, seed=NULL_SEED)},
                               params={'n': N_NULL, 'seed': NULL_SEED}, inputs={'labels': labels_.to_numpy(), 'calls': calls,
                                                                                'drivers': digest(json.dumps(drivers, sort_keys=True))})['null']
            ari = adjusted_rand_score(labels_.reindex(shared), groups69[g].reindex(shared)) if len(shared) >= 2 else np.nan
            relabeled = [GROUPS[ds][m][l][g].nunique() if len(GROUPS[ds][m][l][g]) else 0 for l in labels[1:]]
            k = labels_.nunique()
            names_ = NAMES[ds][m][g]
            GROUP_ROWS.append({'dataset': d['title'], 'method': METHODS[m], 'm': m, 'grouping': g, 'calls': len(calls), 'programs': k,
                               'one-call programs': int((table.calls == 1).sum()), 'programs of ≥ 3 calls': int((table.calls >= 3).sum()),
                               'programs per call': k / len(calls), 'coherence': coh, 'coherence z': pn.null_z(coh, null[:, 0]),
                               'replication': rep, 'replication z': pn.null_z(rep, null[:, 1]),
                               'replication, independent pairs (post hoc)': rep_pair,
                               'replication z, independent pairs (post hoc)': pn.null_z(rep_pair, null_pair[:, 1]),
                               'reconstruction ARI': ari, 'calls in both runs': len(shared),
                               'relabeled programs': ', '.join(str(x) for x in relabeled), 'program NCDR': float(np.mean(relabeled)) / k,
                               'named by an ancestor': float(np.mean([names_[p][1] for p in table.index]))})
            table = table.assign(name=[names_[p][0] for p in table.index], named=[names_[p][1] for p in table.index])
            seg = pn.grid_segments(MODELS[ds]['species'].grid, CUTS[ds])
            table['peak segment'] = seg[np.argmax(profiles, axis=1)]
            table['trough segment'] = seg[np.argmin(profiles, axis=1)]
            table['peak position'] = MODELS[ds]['species'].grid[np.argmax(profiles, axis=1)]
            members = labels_.groupby(labels_).groups
            ct = CALL_TABLES[ds][CALL_TABLES[ds].method.eq(METHODS[m])].set_index('pathway_id')
            sides = GT[ds]['species'].side.to_numpy()
            usable = GT[ds]['species'].usable.to_numpy()
            top, score = [], []
            for p in table.index:
                cnt = Counter(gn for c in members[p] for gn in DRIVERS[ds][m]['species'][c])
                top.append(', '.join(gn for gn, _ in sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))[:5]))
                union = d['index'].get_indexer(sorted(set().union(*[DRIVERS[ds][m]['species'][c] for c in members[p]])))
                union = union[union >= 0]
                score.append(pn.position_score(sides[union[usable[union]]]))
            table['top driver genes'] = top
            table['position score'] = score
            table['position class'] = [pn.score_class(s, SCORE_THRESHOLD) for s in score]
            table['calls (names)'] = [' | '.join(sorted(ct.reindex(list(members[p])).pathway.astype(str))) for p in table.index]
            table = table.rename_axis('program').reset_index().sort_values(['calls', 'program'], ascending=[False, True])
            PROGRAM_TABLES[(ds, m, g)] = (table, profiles)
            table.to_csv(tables / f'programs_{m}_{g}_{ds}.csv', index=False)
GROUPING = pd.DataFrame(GROUP_ROWS)
GROUPING.to_csv(tables / 'grouping_criteria.csv', index=False)

SETTING_ROWS = []
for ds, d in DATA.items():
    for m in METHODS:
        calls = sorted(CALLS[ds][m]['species'])
        if len(calls) < 2:
            continue
        base = GROUPS[ds][m]['species']
        for th in SWEEP_THRESHOLDS:
            for res in SWEEP_RESOLUTIONS:
                alt = build_groupings(d['sets'], calls, DRIVERS[ds][m]['species'], BESTP[ds][m]['species'], SHAPE[ds]['species'], d['index'],
                                      STAT71[ds]['species'].z, REDUNDANCY[ds].representative, threshold=th, resolution=res, only='NET')['NET']
                SETTING_ROWS.append({'dataset': d['title'], 'method': METHODS[m], 'grouping': 'NET', 'threshold': th, 'resolution': res,
                                     'programs': alt.nunique(), 'ARI against the default': adjusted_rand_score(base['NET'].reindex(calls), alt.reindex(calls))})
        for cut in SWEEP_CUTS:
            alt = build_groupings(d['sets'], calls, DRIVERS[ds][m]['species'], BESTP[ds][m]['species'], SHAPE[ds]['species'], d['index'],
                                  STAT71[ds]['species'].z, REDUNDANCY[ds].representative, cut=cut, only='OVL')['OVL']
            SETTING_ROWS.append({'dataset': d['title'], 'method': METHODS[m], 'grouping': 'OVL', 'cut': cut, 'programs': alt.nunique(),
                                 'ARI against the default': adjusted_rand_score(base['OVL'].reindex(calls), alt.reindex(calls))})
SETTINGS = pd.DataFrame(SETTING_ROWS)
SETTINGS.to_csv(tables / 'grouping_settings.csv', index=False)
SETTING_SUMMARY = SETTINGS.groupby(['dataset', 'method', 'grouping'])['ARI against the default'].agg(['median', 'min', 'max'])

GV = []
for (ds_title, method), block in GROUPING.groupby(['dataset', 'method'], sort=False):
    rows_ = block.set_index('grouping')
    for a, b in (('NET', 'OVL'), ('NET', 'RED'), ('NET-pos', 'OVL')):
        directions = {k: v for k, v in GROUP_DIRECTIONS.items() if not (k == 'coherence z' and 'NET-pos' in (a, b))}
        GV.append({'dataset': ds_title, 'method': method, 'comparison': f'{a} against {b}',
                   'verdict': pn.dominance(rows_.loc[a].to_dict(), rows_.loc[b].to_dict(), directions, GROUP_TOL),
                   **{k: f'{rows_.loc[a, k]:.3f}; {rows_.loc[b, k]:.3f}' for k in GROUP_DIRECTIONS}})
GROUP_VERDICTS = pd.DataFrame(GV)
GROUP_VERDICTS.to_csv(tables / 'grouping_verdicts.csv', index=False)
display(GROUPING.set_index(['dataset', 'method', 'grouping'])[['calls', 'programs', 'programs of ≥ 3 calls', 'programs per call', 'coherence',
                                                               'coherence z', 'replication', 'replication z', 'reconstruction ARI',
                                                               'program NCDR', 'named by an ancestor']].round(3))
display(SETTING_SUMMARY.round(3))
display(GROUP_VERDICTS)
print(f'Q2 {time.perf_counter() - t0:.0f} s')

## 6 · Q3: discovery

- **Program tables** (`programs_{method}_{grouping}_{ds}.csv`): name, calls, driver genes, peak and trough segments,
  position class, coherence, replication and member calls. The largest programs of F's calls under NET are shown
  below.
- **Union network:** calls F ∪ C ∪ N (condition), driver genes by F's rule for every set, NET's settings. For each
  program, the methods that call at least one of its calls; programs found by each method, by pairs and by all three,
  and the program Jaccard for each pair.

In [ ]:
SHOW = ['program', 'calls', 'name', 'peak segment', 'trough segment', 'position class', 'coherence', 'replication', 'top driver genes']
for ds in DATASETS:
    print(DATA[ds]['title'], '· positional F-test calls, NET programs')
    display(PROGRAM_TABLES[(ds, 'F', 'NET')][0][SHOW].head(12).round(2))

UNION, UNION_ROWS, UNION_OVERLAP = {}, [], []
for ds, d in DATA.items():
    calls = sorted(set().union(*[CALLS[ds][m]['species'] for m in METHODS]))
    drivers = f_drivers(ds, calls, STAT71[ds]['species'], GT[ds]['species'], d['sets'])
    p = PRIMARY[ds]['species'].p_one_sided.astype(float)
    names, J = pn.driver_jaccard({c: drivers[c] for c in calls})
    labels_ = pn.leiden_programs(names, J, threshold=JACCARD_EDGE, resolution=RESOLUTION, seed=LEIDEN_SEED, best_p=p.reindex(names).to_numpy())
    by_prog = {k: list(v.index) for k, v in labels_.groupby(labels_)}
    degree = dict(zip(names, ((J >= JACCARD_EDGE) & ~np.eye(len(names), dtype=bool)).sum(axis=1)))
    names_ = program_names(by_prog, p, degree)
    UNION[ds] = {'calls': calls, 'labels': labels_, 'drivers': drivers, 'J': J, 'names': names_}
    found = {m: {k for k, v in by_prog.items() if set(v) & CALLS[ds][m]['species']} for m in METHODS}
    for k, v in by_prog.items():
        UNION_ROWS.append({'dataset': d['title'], 'program': k, 'calls': len(v), 'name': names_[k][0],
                           **{f'{METHODS[m]} calls': len(set(v) & CALLS[ds][m]['species']) for m in METHODS},
                           'found by': ' + '.join(m for m in METHODS if k in found[m]), 'members': ' | '.join(d['names'][c] for c in sorted(v))})
    combos = Counter(' + '.join(m for m in METHODS if k in found[m]) for k in by_prog)
    UNION_OVERLAP.append({'dataset': d['title'], 'calls': len(calls), 'programs': len(by_prog),
                          **{f'programs found by {m}': len(found[m]) for m in METHODS},
                          **{f'Jaccard {a}-{b} (programs)': jaccard(found[a], found[b]) for a, b in (('F', 'C'), ('F', 'N'), ('C', 'N'))},
                          **{f'Jaccard {a}-{b} (sets)': jaccard(CALLS[ds][a]['species'], CALLS[ds][b]['species']) for a, b in (('F', 'C'), ('F', 'N'), ('C', 'N'))},
                          **{f'programs found by {c} only' if '+' not in c else f'programs found by {c}': combos.get(c, 0)
                             for c in ('F', 'C', 'N', 'F + C', 'F + N', 'C + N', 'F + C + N')}})
UNION_PROGRAMS = pd.DataFrame(UNION_ROWS)
for ds in DATASETS:
    UNION_PROGRAMS[UNION_PROGRAMS.dataset.eq(DATA[ds]['title'])].to_csv(tables / f'union_programs_{ds}.csv', index=False)
UNION_TABLE = pd.DataFrame(UNION_OVERLAP)
UNION_TABLE.to_csv(tables / 'union_overlap.csv', index=False)
display(UNION_TABLE.T)

## 7 · Application: notebook 91's injury screen (descriptive)

NET on 91's level-up, level-down and location sets separately. Driver genes: members with `level change` (and the
set's sign of L) for level sets, `location change` for location sets. A call's peak is 91's set `peak` (91's segment
units). Reported: programs, their names, and the share of calls in programs of ≥ 3 calls.

In [ ]:
S91 = pd.read_csv(T91 / 'tables' / 'i5_library_sets.csv').set_index('set')
G91 = pd.read_csv(T91 / 'tables' / 'i_genes.csv.gz').set_index('gene')
SCREEN, SCREEN_ROWS = {}, []
sets_aki = DATA[aki]['sets']
for cls, mask, flag, sign in (('level up', S91['level set'].astype(bool) & S91.L.gt(0), 'level change', 1),
                              ('level down', S91['level set'].astype(bool) & S91.L.lt(0), 'level change', -1),
                              ('location', S91['location set'].astype(bool), 'location change', 0)):
    calls = sorted(S91.index[mask])
    genes_ok = G91.index[G91[flag].astype(bool) & ((np.sign(G91.L) == sign) if sign else True)]
    drivers = {c: set(sets_aki[c]) & set(genes_ok) for c in calls}
    pcol = 'p L' if flag == 'level change' else 'p M'
    names, J = pn.driver_jaccard(drivers)
    labels_ = pn.leiden_programs(names, J, threshold=JACCARD_EDGE, resolution=RESOLUTION, seed=LEIDEN_SEED,
                                 best_p=S91[pcol].reindex(names).to_numpy())
    by_prog = {k: list(v.index) for k, v in labels_.groupby(labels_)}
    degree = dict(zip(names, ((J >= JACCARD_EDGE) & ~np.eye(len(names), dtype=bool)).sum(axis=1)))
    names_ = program_names(by_prog, S91[pcol], degree)
    SCREEN[cls] = {'calls': calls, 'labels': labels_, 'J': J, 'names': names_, 'drivers': drivers,
                   'no drivers': sum(not drivers[c] for c in calls)}
    for k, v in by_prog.items():
        cnt = Counter(g for c in v for g in drivers[c])
        SCREEN_ROWS.append({'class': cls, 'program': k, 'calls': len(v), 'name': names_[k][0], 'named by an ancestor': names_[k][1],
                            'median peak (91 units)': float(S91.peak.reindex(v).median()),
                            'driver genes': len(set().union(*[drivers[c] for c in v])),
                            'top driver genes': ', '.join(g for g, _ in sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))[:5]),
                            'members': ' | '.join(sorted(S91.name.reindex(v).astype(str)))})
SCREEN_PROGRAMS = pd.DataFrame(SCREEN_ROWS)
SCREEN_PROGRAMS.to_csv(tables / 'screen91_programs.csv', index=False)
SCREEN_SUMMARY = SCREEN_PROGRAMS.groupby('class').apply(lambda b: pd.Series({
    'calls': int(b.calls.sum()), 'programs': len(b), 'programs of ≥ 3 calls': int((b.calls >= 3).sum()),
    'share of calls in programs of ≥ 3': float(b.calls[b.calls >= 3].sum() / b.calls.sum()),
    'named by an ancestor': float(b['named by an ancestor'].mean()),
    'calls without driver genes': SCREEN[b.name]['no drivers']}), include_groups=False)
display(SCREEN_SUMMARY.round(3))
for cls in SCREEN:
    print(cls)
    display(SCREEN_PROGRAMS[SCREEN_PROGRAMS['class'].eq(cls)].sort_values('calls', ascending=False)
            [['program', 'calls', 'name', 'median peak (91 units)', 'top driver genes']].head(10).round(2))

## 8 · Figures

- **93a:** the NET network of the positional F-test's calls on each dataset; node colour = where the case is
  relatively highest along the PT, marker = position class, size = driver genes; programs of ≥ 3 calls are shaded.
- **93b:** program profiles (the mean positional difference of the program's driver genes) of the largest programs,
  with each case specimen's profile.
- **93c:** the grouping criteria; **93d:** the calling criteria.
- **93e:** the union network coloured by the methods that call each set.
- **93f:** notebook 91's screen networks.

In [ ]:
import igraph as ig
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_rgba
from matplotlib.lines import Line2D
from scipy.spatial import ConvexHull

POSITION_CMAP = LinearSegmentedColormap.from_list('pt', ['#3B4CC0', '#9BB8E8', '#E8D9A0', '#E08A4A', '#B40426'])
MARKERS = {'toward S3': '^', 'toward S1': 'v', 'mixed': 'o'}


def short(text, n=34):
    text = str(text)
    return text if len(text) <= n else text[:n - 1] + '…'


def layout_for(J, labels, threshold, seed=LEIDEN_SEED):
    """scoranet's grouped layout: Fruchterman-Reingold with edges between programs at 1% weight."""
    n = J.shape[0]
    iu, ju = np.triu_indices(n, 1)
    keep = J[iu, ju] >= threshold
    lab = np.asarray(labels)
    w = J[iu, ju][keep] * np.where(lab[iu[keep]] == lab[ju[keep]], 1., .01)
    g = ig.Graph(n=n, edges=list(zip(iu[keep].tolist(), ju[keep].tolist())))
    random.seed(seed)
    coords = np.array(g.layout_fruchterman_reingold(weights=list(w) if len(w) else None, niter=2000).coords) if n > 1 else np.zeros((n, 2))
    return coords, iu[keep], ju[keep], J[iu, ju][keep]


def place_labels(anchors, span, height):
    """Nudge label anchors (x, y) upward or downward until no two labels of width ``span`` overlap."""
    placed = []
    for x, y in anchors:
        for step in range(40):
            dy = (step + 1) // 2 * height * (1 if step % 2 else -1)
            yy = y + dy
            if all(abs(x - px) > span or abs(yy - py) > height for px, py in placed):
                break
        placed.append((x, yy))
    return placed


def draw_network(ax, coords, ei, ej, w, colors, markers, sizes, labels, names, *, label_top=8, hull_min=3, hulls=True):
    for a, b, ww in zip(ei, ej, w):
        ax.plot(coords[[a, b], 0], coords[[a, b], 1], color=LIGHT, lw=.3 + 1.2 * ww, zorder=1)
    lab = pd.Series(labels)
    counts = lab.value_counts()
    chosen = list(counts.index[counts >= hull_min][:label_top])
    anchors = []
    for k in chosen:
        pts = coords[np.flatnonzero(lab.to_numpy() == k)]
        if hulls and len(pts) >= 3:
            try:
                hull = ConvexHull(pts)
                poly = pts[hull.vertices]
                centre = poly.mean(axis=0)
                poly = centre + (poly - centre) * 1.15
                ax.fill(poly[:, 0], poly[:, 1], color='#efeeea', zorder=0, lw=0)
            except Exception:
                pass
        top = pts[np.argmax(pts[:, 1])]
        anchors.append((top[0], top[1] + .03 * (np.ptp(coords[:, 1]) or 1.)))
    width = np.ptp(coords[:, 0]) if len(coords) else 1.
    height = np.ptp(coords[:, 1]) if len(coords) else 1.
    for k, (x, y) in zip(chosen, place_labels(anchors, .42 * width, .05 * height)):
        ax.text(x, y, short(names[k][0], 30), fontsize=5, color=MUTED, ha='center', va='bottom', zorder=4,
                bbox=dict(boxstyle='square,pad=0.05', fc='white', ec='none', alpha=.7))
    for mk in set(markers):
        idx = [i for i, x in enumerate(markers) if x == mk]
        ax.scatter(coords[idx, 0], coords[idx, 1], s=np.asarray(sizes)[idx], c=np.asarray(colors)[idx], marker=mk,
                   edgecolors=INK, linewidths=.25, zorder=3)
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ('left', 'bottom'):
        ax.spines[side].set_visible(False)


fig = plt.figure(figsize=(180 * MM, 95 * MM))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, .04], wspace=.08)
for col, ds in enumerate(DATASETS):
    d = DATA[ds]
    calls = sorted(CALLS[ds]['F']['species'])
    names, J = pn.driver_jaccard({c: DRIVERS[ds]['F']['species'][c] for c in calls})
    labels_ = GROUPS[ds]['F']['species']['NET'].reindex(names).to_numpy()
    coords, ei, ej, w = layout_for(J, labels_, JACCARD_EDGE)
    ct = CALL_TABLES[ds][CALL_TABLES[ds].method.eq(METHODS['F'])].set_index('pathway_id').reindex(names)
    grid = MODELS[ds]['species'].grid
    norm_ = Normalize(grid[0], grid[-1])
    colors = [POSITION_CMAP(norm_(x)) for x in ct['peak position']]
    ax = fig.add_subplot(gs[0, col])
    draw_network(ax, coords, ei, ej, w, colors, [MARKERS[c] for c in ct['position class']], 6 + 3 * np.sqrt(ct['usable driver genes']),
                 labels_, NAMES[ds]['F']['NET'])
    ax.set_title(f'{d["title"]} · positional F-test calls ({len(calls)})', loc='left')
cax = fig.add_subplot(gs[0, 2])
sm = plt.cm.ScalarMappable(cmap=POSITION_CMAP, norm=Normalize(0, 1))
cb = fig.colorbar(sm, cax=cax)
cb.set_ticks([0, 1], labels=['PT start', 'PT end'])
cb.set_label('peak of the case − reference difference')
fig.legend(handles=[Line2D([], [], marker=v, color='none', markerfacecolor='#cccccc', markeredgecolor=INK, markersize=4, label=k)
                    for k, v in MARKERS.items()], loc='lower left', ncol=3, bbox_to_anchor=(.12, -.02))
save(fig, 'fig93a_networks_ftest')
plt.show()

In [ ]:
import textwrap

N_PROFILES = 6
fig, axes = plt.subplots(2, N_PROFILES, figsize=(180 * MM, 78 * MM), sharex='row')
for r, ds in enumerate(DATASETS):
    d = DATA[ds]
    table, profiles = PROGRAM_TABLES[(ds, 'F', 'NET')]
    grid = MODELS[ds]['species'].grid
    stats = STATS[(ds, 'F')]
    labels_ = GROUPS[ds]['F']['species']['NET']
    order = table.program.to_numpy()[:N_PROFILES]
    _, mean = stats.per_program(labels_)
    prog_index = list(stats.per_program(labels_)[0].index)
    spec_prof = []
    uniq, _, U = stats._unions(labels_)
    m_ = np.asarray(U @ stats._rep_ok).ravel()
    for s in stats._rep:
        spec_prof.append(np.divide(np.asarray(U @ s), m_[:, None], out=np.full((len(uniq), s.shape[1]), np.nan), where=m_[:, None] > 0))
    for c, prog in enumerate(order):
        ax = axes[r, c]
        i = prog_index.index(prog)
        for cut in CUTS[ds]:
            ax.axvline(cut, color=LIGHT, lw=.6, zorder=0)
        ax.axhline(0, color=LIGHT, lw=.5, zorder=0)
        for s, (prof, name) in enumerate(zip(spec_prof, d['case_specimens'])):
            ax.plot(grid, prof[list(uniq).index(prog)], color=CASE_COLOR, lw=.6, alpha=.55 if s else .9, ls='-' if s == 0 else '--')
        ax.plot(grid, mean[i], color=INK, lw=1.1)
        inner = slice(int(.05 * len(grid)), int(.95 * len(grid)) + 1)
        shown = np.concatenate([mean[i][inner]] + [p_[list(uniq).index(prog)][inner] for p_ in spec_prof])
        lo, hi = np.nanmin(shown), np.nanmax(shown)
        pad = .15 * max(hi - lo, .05)
        ax.set_ylim(lo - pad, hi + pad)
        row = table.set_index('program').loc[prog]
        ax.set_title(textwrap.fill(f'{short(row["name"], 44)} ({int(row.calls)})', 24), loc='left', fontsize=5.5)
        if c == 0:
            ax.set_ylabel(f'{d["case_name"]} − {d["reference_name"]}\npositional log2 difference')
        if r == 1:
            ax.set_xlabel('position along the PT')
fig.legend(handles=[Line2D([], [], color=INK, lw=1.1, label='program (both case specimens)'),
                    Line2D([], [], color=CASE_COLOR, lw=.6, label='case specimen 1'),
                    Line2D([], [], color=CASE_COLOR, lw=.6, ls='--', alpha=.55, label='case specimen 2')],
           loc='lower center', ncol=3, bbox_to_anchor=(.5, -.06))
fig.tight_layout()
save(fig, 'fig93b_program_profiles')
plt.show()

In [ ]:
crit = ['programs per call', 'coherence z', 'replication z', 'reconstruction ARI', 'program NCDR']
fig, axes = plt.subplots(1, len(crit), figsize=(180 * MM, 60 * MM), sharey=True)
rows_ = [(ds, m) for ds in DATASETS for m in METHODS]
ylabels = [f'{SHORT_DS[ds]} · {METHODS[m]}' for ds, m in rows_]
offset = {'NET': -.27, 'NET-pos': -.09, 'OVL': .09, 'RED': .27}
for ax, k in zip(axes, crit):
    for y, (ds, m) in enumerate(rows_):
        block = GROUPING[GROUPING.dataset.eq(DATA[ds]['title']) & GROUPING.m.eq(m)].set_index('grouping')
        for g in GROUPINGS:
            if g in block.index and np.isfinite(block.loc[g, k]):
                hollow = (k == 'coherence z' and g == 'NET-pos')
                ax.scatter(block.loc[g, k], y + offset[g], s=12, color='white' if hollow else GROUPING_COLOR[g],
                           edgecolors=GROUPING_COLOR[g], linewidths=.7, zorder=3)
    if 'z' in k:
        ax.axvline(0, color=LIGHT, lw=.6, zorder=0)
    ax.set_title(k, loc='left')
    ax.set_yticks(range(len(rows_)), ylabels)
    ax.invert_yaxis()
fig.legend(handles=[Line2D([], [], marker='o', color='none', markerfacecolor=GROUPING_COLOR[g], markeredgecolor=GROUPING_COLOR[g],
                           markersize=4, label=GROUPINGS[g]) for g in GROUPINGS], loc='lower center', ncol=4, bbox_to_anchor=(.5, -.08))
fig.tight_layout()
save(fig, 'fig93c_grouping_criteria')
plt.show()

In [ ]:
crit = ['swap rule', 'reconstruction Jaccard', 'probe exposure', 'hub robustness']
fig, axes = plt.subplots(1, len(crit), figsize=(150 * MM, 45 * MM))
for ax, k in zip(axes, crit):
    for j, ds in enumerate(DATASETS):
        for i, m in enumerate(METHODS):
            v = CRIT[ds][m].get(k, np.nan)
            if not np.isfinite(v):
                continue
            ax.scatter(i + (j - .5) * .3, v, s=16, color=METHOD_COLOR[m], marker='o' if ds == hvm else 's', edgecolors=INK, linewidths=.3, zorder=3)
    ax.set_xticks(range(len(METHODS)), [METHODS[m] for m in METHODS], rotation=30, ha='right')
    ax.set_title(k, loc='left')
    if k == 'swap rule':
        ax.set_yticks([0, 1], ['fail', 'pass'])
        ax.set_ylim(-.3, 1.3)
    else:
        ax.set_ylim(0, 1.02)
fig.legend(handles=[Line2D([], [], marker='o', color='none', markerfacecolor='#cccccc', markeredgecolor=INK, markersize=4, label='human vs mouse'),
                    Line2D([], [], marker='s', color='none', markerfacecolor='#cccccc', markeredgecolor=INK, markersize=4, label='AKI vs control')],
           loc='lower center', ncol=2, bbox_to_anchor=(.5, -.12))
fig.tight_layout()
save(fig, 'fig93d_calling_criteria')
plt.show()

In [ ]:
COMBO_COLOR = {'F': METHOD_COLOR['F'], 'C': METHOD_COLOR['C'], 'N': METHOD_COLOR['N'], 'F + C': '#7d9a2e', 'F + N': '#22818f',
               'C + N': '#8b8f6a', 'F + C + N': INK}
fig, axes = plt.subplots(1, 2, figsize=(180 * MM, 90 * MM))
for ax, ds in zip(axes, DATASETS):
    u = UNION[ds]
    labels_ = u['labels'].reindex(u['calls']).to_numpy()
    coords, ei, ej, w = layout_for(u['J'], labels_, JACCARD_EDGE)
    combo = [' + '.join(m for m in METHODS if c in CALLS[ds][m]['species']) for c in u['calls']]
    draw_network(ax, coords, ei, ej, w, [COMBO_COLOR[c] for c in combo], ['o'] * len(combo),
                 [6 + 3 * np.sqrt(len(u['drivers'][c])) for c in u['calls']], labels_, u['names'], label_top=8, hulls=False)
    ax.set_title(f'{DATA[ds]["title"]} · all calls ({len(u["calls"])})', loc='left')
fig.legend(handles=[Line2D([], [], marker='o', color='none', markerfacecolor=v, markeredgecolor=INK, markersize=4,
                           label=' + '.join(METHODS[x] for x in k.split(' + '))) for k, v in COMBO_COLOR.items()],
           loc='lower center', ncol=4, bbox_to_anchor=(.5, -.08))
save(fig, 'fig93e_union_network')
plt.show()

In [ ]:
fig = plt.figure(figsize=(180 * MM, 75 * MM))
gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, .04], wspace=.06)
peak_norm = Normalize(float(S91.peak.min()), float(S91.peak.max()))
for col, cls in enumerate(SCREEN):
    s = SCREEN[cls]
    labels_ = s['labels'].reindex(s['calls']).to_numpy()
    coords, ei, ej, w = layout_for(s['J'], labels_, JACCARD_EDGE)
    colors = [POSITION_CMAP(peak_norm(x)) for x in S91.peak.reindex(s['calls'])]
    ax = fig.add_subplot(gs[0, col])
    draw_network(ax, coords, ei, ej, w, colors, ['o'] * len(colors), [4 + 2 * np.sqrt(max(len(s['drivers'][c]), 1)) for c in s['calls']],
                 labels_, s['names'], label_top=6, hulls=False)
    ax.set_title(f'AKI screen · {cls} sets ({len(s["calls"])})', loc='left')
cax = fig.add_subplot(gs[0, 3])
cb = fig.colorbar(plt.cm.ScalarMappable(cmap=POSITION_CMAP, norm=peak_norm), cax=cax)
cb.set_label('set location peak (segment units)')
save(fig, 'fig93f_screen_networks')
plt.show()

## 9 · How the protocol was read, summary and run record

In [ ]:
READING = [
    'Row N tests and corrects only the sets with 10–300 members among the partition\'s usable genes; its calibration p '
    'are pooled over those sets.',
    'dominance(): when every defined criterion is within its tolerance the verdict is "equal" (the protocol\'s '
    '"otherwise mixed" read as equal for this case).',
    'The ancestor share counts all of a program\'s calls in the denominator (Hallmark calls and unmatched sets '
    'included), as the protocol states; scoranet drops calls without an id from the denominator.',
    'Hub names: for NET, the call with the highest degree in the thresholded network, then the smallest p; for the '
    'other groupings, the smallest p.',
    'Notebook 69\'s cuts use 69\'s positions and labels of the reference units in both runs (71\'s unit-matching '
    'table).',
    'C\'s driver genes come from the matched ORA rows with q < 0.05 (the overlap genes listed by the package).',
]
for line in READING:
    print('·', line)
pd.Series(READING, name='reading').to_csv(tables / 'how_the_protocol_was_read.csv', index=False)

RUN_RECORD = {
    'notebook': '93_pt_pathway_networks.ipynb', 'logic': NOTEBOOK_LOGIC_VERSION, 'protocol commit': 'dbefcba',
    'guards': guard_log, 'seconds': round(time.perf_counter() - NOTEBOOK_START, 1), 'stage seconds': SECONDS,
    'versions': {'igraph': ig.__version__, 'pseudospace_reconstruction': EXTERNAL},
    'hierarchies': HIERARCHY_SHA, 'reactome pathways (human)': REACTOME.size, 'kegg maps': KEGG.size,
    'parameters': {'alpha': ALPHA, 'half_range': HALF_RANGE, 'jaccard_edge': JACCARD_EDGE, 'resolution': RESOLUTION,
                   'leiden_seed': LEIDEN_SEED, 'profile_edge': PROFILE_EDGE, 'n_null': N_NULL, 'null_seed': NULL_SEED,
                   'n_hub': N_HUB, 'min_share': MIN_SHARE, 'max_descendant_share': MAX_DESC_SHARE}}
(output / 'run_record.json').write_text(json.dumps(RUN_RECORD, indent=1, default=str))
print(f'Done in {RUN_RECORD["seconds"]:.0f} s')